# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `songhyodoc/connect-ai date` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/Llama-3.2-1B-Instruct`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `songhyodoc/llama-1baimentory` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 40 · seq 1024 · linear · 양자화 q4_k_m (데이터 10개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "eyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLrs7Trno/ruZsg7IaM6rCAIOyYqOuLpCAoUHVycGxlIENvdykg7Ja065a76rKMIO2VtD8ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiIyDrs7Trno/ruZsg7IaM6rCAIOyYqOuLpCAoUHVycGxlIENvdylcbiMg67O0656P67mbIOyGjOqwgCDsmKjri6QgKFB1cnBsZSBDb3cpIOKAlCDrp4jsvIDtjIUg7JmE7KCEIOygleumrFxuXG7shLjsiqQg6rOg65SYKFNldGggR29kaW4p7J2YIOuniOy8gO2MhSDqs6DsoIQuIO2VnCDrrLjsnqU6IFwi7Y+J67KU7ZWY66m0IOuztOydtOyngCDslYrripTri6QuIOyjvOuqqe2VoCDrp4ztlZjqsowocmVtYXJrYWJsZSkg66eM65Ok7Ja06528LlwiXG5cbiMjIDEuIOuztOuej+u5myDshowgPSDrpqzrp4jsu6TruJRcbi0g7Y+J67KU7ZWcIOyGjCDsiJjrsLEg66eI66as64qUIOyViCDrs7Tsnbjri6QuIOuztOuej+u5myDshozripQg64iE6rWs64KYIOupiOy2sOyEnCDrs7Tqs6Ag7Lmc6rWs7JeQ6rKMIOunkO2VnOuLpC5cbi0g66as66eI7Luk67iUID0gXCJyZW1hcmso7Ja46riJKe2VoCDrp4ztlZxcIi4g66eI7LyA7YyF7J2YIO2VteyLrOydgCDqtJHqs6Ag6riw7Iig7J20IOyVhOuLiOudvCDsoJztkogg7J6Q7LK06rCAIOyjvOuqqe2VoCDrp4ztlZzqsIDri6QuXG4tIOumrOuniOy7pOu4lOydmCDrsJjrjIDrp5DsnYAgXCLrgpjsgahcIuydtCDslYTri4jrnbwgXCLrp6TsmrAg7KKL7J2MKHZlcnkgZ29vZClcIi4g66y064Kc7ZWY6rOgIOyViOyghO2VnCDqsoPsnYAg67O07J207KeAIOyViuuKlOuLpC5cblxuIyMgMi4g7JibIOuniOy8gO2MheydmCDso73snYwg4oCUIFRWwrfsgrDsl4Ug67O17ZWp7LK0XG4tIO2PieuylO2VnCDsoJztkoggKyDrp4nrjIDtlZwg6rSR6rOg67mEID0g66ek7LacLCDsnbQg6rO17Iud7J20IOustOuEiOyhjOuLpC5cbi0g7IKs656M65Ok7J2AIOq0keqzoOulvCDrrLTsi5ztlZjripQg67KV7J2EIOuwsOyboOuLpC4g6rSR6rOg66GcIO2PieuylO2VnCDsoJztkojsnYQg6rWs7ZWgIOyImCDsl4bri6QuXG4tIOuniOy8gO2MheydhCDsoJztkogg64Gd7JeQIOuNp+u2meydtOyngCDrp5Dqs6AsIOygnO2SiCDslYjsl5Ag64K07J6l7ZWY6528LlxuXG4jIyAzLiDsg4jroZzsmrQgUCDigJQgUHVycGxlIENvd1xuLSDsoITthrUg66eI7LyA7YyFIFAoUHJvZHVjdC9QcmljZS9Qcm9tb3Rpb24vUG9zaXRpb25pbmfigKYp7JeQICdQdXJwbGUgQ293J+ulvCDrjZTtlZjrnbwuXG4tIOq4sO2ajSDssqsg64uo6rOE67aA7YSwIFwi7J206rKMIOyZnCDsnoXshozrrLgg64Kg6rmMP1wi66W8IOyEpOqzhOyXkCDrhKPslrTrnbwuXG5cbiMjIDQuIOuIhOq1rOyXkOqyjCDigJQg7Jik7YOA7L+gKE90YWt1KVxuLSDrqqjrkZDrpbwg66eM7KGx7Iuc7YKk66Ck64qUIOygnO2SiOydgCDslYTrrLTrj4Qg7KO866qp7ZWY7KeAIOyViuuKlOuLpC5cbi0g7Jik7YOA7L+gID0g7Ja065akIOqyg+yXkCDruYTsoJXsg4HsoIHsnLzroZwg7Je06rSR7ZWY64qUIOyGjOyImC4g64+Iwrfsi5zqsITsnYQg7JOw6rOgIOuCqOyXkOqyjCDrlqDrk6Dri6QuXG4tIOuvuOyngOq3vO2VnCDri6TsiJjrs7Tri6Qg7Je06rSR7ZWY64qUIOyGjOyImOulvCDrhbjroKTrnbwuIOq3uOuTpOydtCDsi5zsnqXsnYQg7Jew64ukLlxuXG4jIyA1LiDslrTrlrvqsowg7Y287KeA64KYIOKAlCDsiqTri4jsoIAoU25lZXplcinsmYAg7JWE7J2065SU7Ja0IOuwlOydtOufrOyKpFxuLSDsiqTri4jsoIAgPSDslYTsnbTrlJTslrTrpbwg7J6s7LGE6riw7ZWY65OvIO2NvOucqOumrOuKlCDsoITtjIzsnpAuIOyeheyGjOusuOydmCDtlbXsi6wuXG4tIOumrOuniOy7pOu4lO2VnCDqsoPrp4wg7KCE7YyM65Cc64ukLiDtj4nrspTtlZwg6rG0IOyVhOustOuPhCDsuZzqtazsl5Dqsowg66eQ7ZWY7KeAIOyViuuKlOuLpC5cbi0g6rSR6rOg67mEIOuMgOyLoCwg7Iqk64uI7KCA6rCAIOyekOuwnOyggeycvOuhnCDtjbzrnKjrprQg7J207Jyg66W8IOygnO2SiOyXkCDsi6zslrTrnbwuIO2NvOucqOumrOq4sCDsib3qsowg66eM65Ok7Ja06528LlxuXG4jIyA2LiDslrzrpqzslrTri7XthLDsmYAg7LqQ7KaYXG4tIOuMgOykkeydhCDsp4HsoJEg64W466as7KeAIOuniOudvC4g7Ja866as7Ja064u17YSw66W8IOuFuOumrOqzoCDqt7jrk6TsnbQg64uk7IiY7JeQ6rKMIOyghO2MjO2VmOqyjCDtlZjrnbwuXG4tIOumrOuniOy7pOu4lO2VmOyngCDslYrsnLzrqbQg7Ja866as7Ja064u17YSw7JmAIOuLpOyImCDsgqzsnbQg7LqQ7KaYKOqwhOq3uSnsnYQg66q7IOuEmOqzoCDsgqzrnbzsp4Tri6QuXG5cbiMjIDcuIOq3ueuLqOycvOuhnCwg6rCA7J6l7J6Q66as66GcXG4tIOyLnOyepSDtlZzqsIDsmrTrjbAo7KSR6rCEIO2SiOyniMK36rCA6rKpwrfrrLTrgpztlagp64qUIOqwgOyepSDrtpDruYTqs6Ag6rCA7J6lIOyViCDrs7Tsnbjri6QuXG4tIO2VnCDstpXsl5DshJwg6re564uo7Jy866GcOiDqsIDsnqUg67mg66W4L+yLvC/qs6DquIkv64uo7Iic7ZWcL+yghOusuOyggeyduC4g7Ja07KSR6rCE7ZWo7J20IOqwgOyepSDsnITtl5jtlZjri6QuXG5cbiMjIDguIOuRkOugpOybgOydtCDtj4nrspTtlajsnYQg66eM65Og64ukXG4tIOumrOuniOy7pOu4lO2VmOyngCDrqrvtlZjripQg7J207Jyg64qUIOu5hO2MkMK37Iuk7Yyo6rCAIOuRkOugpOybjOyEnC4g7JWI7KCEIn1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6Ik1yQmVhc3Qg7ZuE7YK5IOuhnOyngeyXkCDrjIDtlbQg7JWM66Ck7KSYIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IiMgTXJCZWFzdCDtm4Ttgrkg66Gc7KeBXG4jIE1yQmVhc3Qg7ZuE7YK5IOuhnOyngSDrtoTshJ1cblxuIyMg7ZW17IusIO2MqO2EtFxuLSAqKuyyqyA17LSIKio6IOy2qeqyqeyggSDtlonrj5nCt+qysOqzvCDrr7jrpqzrs7TquLAgKFwi7Jqw66as64qUIOydtCDsgqzrnozsl5DqsowgMTAw66eMIOuLrOufrOulvCDspKzslrTsmpQuLi5cIilcbi0gKio1fjMw7LSIKio6IOychOq4sCDshKTsoJXCt+ydtO2VtOq0gOqzhCDrqoXsi5wgKFwiLi4u7ZWY7KeA66eMIOyhsOqxtOydtCDsnojso6AuXCIpXG4tICoq6rOg67CA64+EIOy7tyoqOiDtj4nqt6AgMS417LSI64u5IDHsu7csIOyLnOyEoCDrqrsg65a86rKMXG4tICoq7Iir7J6QIOqwleyhsCoqOiDtla3sg4Eg6rWs7LK07KCBIOyImOy5mCAoXCIxMDDrp4wg64us65+sXCIsIFwiMjTsi5zqsIRcIiwgXCI366qFXCIpXG5cbiMjIOyggeyaqSDssrTtgazrpqzsiqTtirhcbi0gWyBdIOyyqyA17LSI7JeQIOqysOqzvCDrr7jrpqzrs7TquLAg7J6I64KYP1xuLSBbIF0g7Iuc7LKt7J6Q6rCAIFwi7J206rKMIOynhOynnD9cIiDsnZjsi6ztlZjqsowg66eM65Oc64KYP1xuLSBbIF0gMzDstIgg7JWI7JeQIOychOq4sMK37J207ZW06rSA6rOEIOuqhe2Zle2VnOqwgD9cbi0gWyBdIOy7tyDtj4nqt6Ag6ri47J206rCAIDLstIgg7J207ZWY7J246rCAPyJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLsiqTtgqw6IPCfjqwg7ZuE7YK5IOu2hOyEneq4sCDsoITrnrXsnbQg662Q7JW8PyJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIOyKpO2CrDog8J+OrCDtm4Ttgrkg67aE7ISd6riwXG7rs7jsnbgg7LGE64SQIOy1nOq3vCDsmIHsg4HsnZgg7LKrIDMw7LSIIO2bhO2CuSDtjKjthLTsnYQg7J6Q64+ZIOu2hOyEnS5cbuyLpO2WiSDqsIDriqXtlZwg7YyM7J207I2sIOyKpO2CrDogPHJ1bj5weXRob24zIFwiQzpcXFVzZXJzXFxQQ1xcRGVza3RvcFxcY29ubmVjdC1haS1wYWNrc1xc7Iqk7YKsXFx5b3V0dWJlXFxob29rX2FuYWx5emVyLnB5XCI8L3J1bj4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiQ29ubmVjdCBBSSDigJQgQnJhaW4tR2l0SHViIOuPmeq4sO2ZlCDslYTtgqTthY3sspgg66CI7Y2865+w7IqkIOyWtOuWu+qyjCDqtaztmITtlbQ/In0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IiMgQ29ubmVjdCBBSSDigJQgQnJhaW4tR2l0SHViIOuPmeq4sO2ZlCDslYTtgqTthY3sspgg66CI7Y2865+w7IqkXG4jIPCfp6AgQ29ubmVjdCBBSSDigJQgQnJhaW4tR2l0SHViIOuPmeq4sO2ZlCDslYTtgqTthY3sspgg66CI7Y2865+w7IqkXG5cbj4gKirsnbQg66y47ISc64qUIOyWtOuWpCBBSeuToCDsnbQg7L2U65Oc67Kg7J207Iqk7J2YIOq5g+2XiOu4jCDrj5nquLDtmZQg6rWs7KGw66W8IOymieyLnCDtjIzslYXtlaAg7IiYIOyeiOuPhOuhnSDrp4zrk6Ag66CI7Y2865+w7Iqk7J6F64uI64ukLioqXG5cbi0tLVxuXG4jIyAxLiDtlITroZzsoJ3tirgg7JyE7LmYIOuwjyDtlbXsi6wg7YyM7J28XG5cbmBgYFxuL1VzZXJzL2pheS/roZzsu6zthYzsiqTtirgvbG9jYWwtYWktY29kZXIvXG7ilJzilIDilIAgcGFja2FnZS5qc29uICAgICAgICAgIOKGkCDshKTsoJUg7Iqk7YKk66eIIChjb250cmlidXRlcy5jb25maWd1cmF0aW9uKVxu4pSc4pSA4pSAIHNyYy9cbuKUgiAgIOKUlOKUgOKUgCBleHRlbnNpb24udHMgICAgICDihpAg66qo65OgIOuhnOyngeydtCDri7TquLQg64uo7J28IO2MjOydvCAoMjYwMCsgbGluZXMpXG7ilJzilIDilIAgb3V0L1xu4pSCICAg4pSU4pSA4pSAIGV4dGVuc2lvbi5qcyAgICAgIOKGkCDruYzrk5wg6rKw6rO866y8IChlc2J1aWxkKVxu4pSc4pSA4pSAIGJyYWluLXZpei5odG1sICAgICAgICDihpAg7KeA7IudIOuEpO2KuOybjO2BrCDsi5zqsIHtmZQgSFRNTFxu4pSU4pSA4pSAIHN5c3RlbV9zY2hlbWEuanNvbiAgICDihpAgQUkg7JeQ7J207KCE7Yq4IOuPhOq1rCDsiqTtgqTrp4hcbmBgYFxuXG4qKu2VteyLrCDtjIzsnbzsnYAg65SxIDLqsJw6Kipcbi0gYHBhY2thZ2UuanNvbmAg4oCUIFZTIENvZGUg7ISk7KCVIOyKpO2CpOuniCDshKDslrhcbi0gYHNyYy9leHRlbnNpb24udHNgIOKAlCDrqqjrk6Ag64+Z6riw7ZmUIOuhnOyngVxuXG4tLS1cblxuIyMgMi4gVlMgQ29kZSDshKTsoJUg7YKkIChDb25maWd1cmF0aW9uKVxuXG5gcGFja2FnZS5qc29uYCDihpIgYGNvbnRyaWJ1dGVzLmNvbmZpZ3VyYXRpb24ucHJvcGVydGllc2Dsl5Ag7ISg7Ja465CoOlxuXG58IOyEpOyglSDtgqQgfCDsmqnrj4QgfCDquLDrs7jqsJIgfFxufC0tLS0tLS0tLXwtLS0tLS18LS0tLS0tLS18XG58IGBjb25uZWN0QWlMYWIubG9jYWxCcmFpblBhdGhgIHwg66Gc7LusIOyngOyLnSDtj7TrjZQg7KCI64yAIOqyveuhnCB8IGBcIlwiYCAo67mE66m0IGB+Ly5jb25uZWN0LWFpLWJyYWluYCkgfFxufCBgY29ubmVjdEFpTGFiLnNlY29uZEJyYWluUmVwb2AgfCDquYPtl4jruIwg7KCA7J6l7IaMIFVSTCB8IGBcIlwiYCB8XG58IGBjb25uZWN0QWlMYWIub2xsYW1hVXJsYCB8IEFJIOyEnOuyhCDso7zshowgfCBgaHR0cDovLzEyNy4wLjAuMToxMTQzNGAgfFxufCBgY29ubmVjdEFpTGFiLmRlZmF1bHRNb2RlbGAgfCBBSSDrqqjrjbgg7J2066aEIHwgYGdlbW1hNDplMmJgIHxcbnwgYGNvbm5lY3RBaUxhYi5yZXF1ZXN0VGltZW91dGAgfCBBSSDsnZHri7Ug64yA6riwIOyLnOqwhCjstIgpIHwgYDMwMGAgfFxuXG4qKuKaoO+4jyDshKTsoJXsnYAgYHZzY29kZS5Db25maWd1cmF0aW9uVGFyZ2V0Lkdsb2JhbGDroZwg7KCA7J6l65CoKiogKOybjO2BrOyKpO2OmOydtOyKpOqwgCDslYTri4wg7KCE7JetKVxuXG4tLS1cblxuIyMgMy4g7ZW17IusIOy9lOuTnCDshLnshZggKGV4dGVuc2lvbi50cyDrgrQg7JyE7LmYKVxuXG4jIyMgMy1BLiDshKTsoJUg7J296riwIO2VqOyImOuTpCAo7IOB64uoKVxuXG5gYGAifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi6rWQ7Jyh7JqpIOyKrOudvOydtOuTnCDtkoAg7IS47Yq4IOKAlCDsi5zsl7Ag7KSR6rCEIOuBvOyasOq4sOyaqSDshKTrqoXtlbTspJgifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiIyDqtZDsnKHsmqkg7Iqs65287J2065OcIO2SgCDshLjtirgg4oCUIOyLnOyXsCDspJHqsIQg64G87Jqw6riw7JqpXG4jIPCfk5og6rWQ7Jyh7JqpIOyKrOudvOydtOuTnCDtkoAg7IS47Yq4IOKAlCDsi5zsl7Ag7KSR6rCEIOuBvOyasOq4sOyaqVxuXG4+IOyLnOyXsCDihpIgXCLsnbTqsowg7Ja065a76rKMIOuQmOuKlCDqsbDslbw/XCIg4oaSIOq1kOycoSDsiqzrnbzsnbTrk5wgMS0y7J6lIOKGkiDri6Tsi5wg7Iuc7JewLiAgXG4+IOqwleydmOyLnSDquYrsnbTroZwg7J6R7ISxLiDtlZnrtoAgU1fCt0FJIOyImOyXhSDsiJjspIDsl5Ag66ee7LakLlxuXG4tLS1cblxuIyDwn46vIFNMSURFIEdST1VQIEEg4oCUIExMTSDquLDrs7gg7JuQ66asICjsi5zsl7AgMiDsoITsl5ApXG5cbiMjIPCfk5ogQS0xLiBcIkFJ6rCAIOyWtOuWu+qyjCDtlZzqta3slrTrpbwg7J207ZW07ZWg6rmMP1wiIChUb2tlbml6YXRpb24pXG5cbioq7YGwIOyniOusuDoqKiBcIkFJ64qUIOyCrOyLpCDri6jslrTrpbwg66qo66aF64uI64ukLiAqKuyIq+yekCoq66eMIOydtO2VtO2VtOyalC5cIlxuXG5gYGBcblwi7JWI64WV7ZWY7IS47JqUIOyCrOyepeuLmFwiXG4gICAgICAgIOKGk1xuICAgVG9rZW5pemVyXG4gICAgICAgIOKGk1xuWzIxMDIsIDQ1MjEsIDE5LCA1NjMwLCAxOTIsIDI4MTRdXG4gICAgICAgIOKGk1xuTExNICjsiJjsi63slrUg6rCcIO2WieugrCDqs7EpXG4gICAgICAgIOKGk1xuW+uLpOydjCDthqDtgbAg7ZmV66WgIOu2hO2PrF1cbiAgICAgICAg4oaTXG5cIuyViOuFle2VmOyEuOyalFwiICjtmZXrpaAgODclKVxuYGBgXG5cbioq7ZW17IusOioqXG4tICoqVG9rZW4qKiA9IOuLqOyWtOqwgCDslYTri4jrnbwgfjN+NOq4gOyekCDsobDqsIEgKO2VnOq1reyWtOuKlCDrs7TthrUgMeq4gOyekCA9IDF+MiDthqDtgbApXG4tIOyCrOuejCDtlZwg66eI65SUID0gMzB+NTAg7Yag7YGwXG4tIExMTeydgCAq64uk7J2MIO2GoO2BsCDtmZXrpaAqIOunjCDsmIjsuKEuIOq3uCDsmbjsl5Qg66qo66aELlxuXG4qKldoeSBpdCBtYXR0ZXJzOioqXG4+IFwiQUnqsIAgKuyZnCDqsIDrgZQg6rGw7KeT66eQ7ZWY64qU7KeAKiDsnbTtlbTtlZjroKTrqbQg4oCUIOq3uOuDpSAq7Ya16rOE7KCB7Jy866GcIOq3uOuftOuTr+2VnCDri6jslrQq66W8IOu9keuKlCDqsbDsmIjsmpQuXCJcblxuLS0tXG5cbiMjIPCfk5ogQS0yLiBcIuyZnCBMTE3snYAgMX4y7LSI66eM7JeQIOuLteydhCDrp4zrk6TquYw/XCIgKEF0dGVudGlvbilcblxuKirtgbAg7KeI66y4OioqIFwiQUnripQg7Ja065a76rKMIDIwMO2GoO2BsCDsnoXroKUg67O06rOgIDXstIgg7JWI7JeQIDUwMO2GoO2BsCDri7XsnYQg66eM65Ok6rmMP1wiXG5cbioqVHJhbnNmb3JtZXIgU2VsZi1BdHRlbnRpb246KipcbmBgYFxuSW5wdXQ6IFwi7IKs7J6l64uYLCDrp6Tstpwg7JWM66Ck7KSYXCJcbiAgICAgICAgIOKGk1xuIOKUjOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUkFxuIOKUgiAgVG9rZW7rp4jri6QgICAgICAgICAgICAg4pSCXG4g4pSCICBcIuyWtOuKkCDri6Trpbgg7Yag7YGw7JeQICAgICAg4pSCXG4g4pSCICAg7KeR7KSR7ZWg6rmMP1wiIOqzhOyCsCAgICAgICDilIJcbiDilIIgICAgICAgICAgICAgICAgICAgICAgIOKUglxuIOKUgiAgXCLrp6TstpxcIiDthqDtgbDsnYAgICAgICAgICAg4pSCXG4g4pSCICDihpIgXCLslYzroKTspJhcIuyXkCA4MCUgICAgICDilIJcbiDilIIgIOKGkiBcIuyCrOyepeuLmFwi7JeQIDE1JSAgICAgIOKUglxuIOKUgiAg4oaSIOuCmOuouOyngOyXkCA1JSAgICAgICAgIOKUglxuIOKUlOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUmFxuICAgICAgICAg4oaTXG4g64uk7J2MIO2GoO2BsCDtmZXrpaAg67aE7Y+sXG5gYGBcblxuKirtlbXsi6w6Kipcbi0gKirrs5HroKwg7LKY66asKiog4oCUIEdQVeqwgCDsiJjsspwg7Yag7YGw7J2EIOuPmeyLnOyXkCDqs4TsgrBcbi0gKipBdHRlbnRpb24qKiA9IFwi7KeA6riIIOyWtOuWpCDri6jslrTqsIAg7KSR7JqU7ZWc6rCAXCIg7J6Q64+ZIO2VmeyKtVxuLSA3QiDrqqjrjbggPSA3MOyWtSDqsJwg7ZaJ66CsIOqzseyFiCJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJDb25uZWN0IEFJIOKAlCAxMDAlIOuhnOy7rOyXkOyEnCDrj4zslYTqsIDripQgMeyduCDquLDsl4Xsl5Ag64yA7ZW0IOyVjOugpOykmCJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIENvbm5lY3QgQUkg4oCUIDEwMCUg66Gc7Lus7JeQ7IScIOuPjOyVhOqwgOuKlCAx7J24IOq4sOyXhVxuIyDwn46sIENvbm5lY3QgQUkg4oCUIDEwMCUg66Gc7Lus7JeQ7IScIOuPjOyVhOqwgOuKlCAx7J24IOq4sOyXhVxuIyMg67Cc7ZGc7JqpIOyKrOudvOydtOuTnCDsvZjthZDsuKAgKOyghOusuCArIOyerOuvuCArIOq1kOycoSBmdXNpb24pXG5cbj4gKirtj6zrp7cqKjog7Iqs65287J2065OcIO2VnCDtjpjsnbTsp4Dri7kg7ZWcIOyEueyFmC4g66eI7YGs64uk7Jq0IOq3uOuMgOuhnCBLZXlub3RlwrdQb3dlclBvaW50wrdOb3Rpb27Ct1NsaWRldsK3TWFycOyXkCDrtpnsl6zrhKPquLAg6rCA64qlLlxuXG4tLS1cblxuIyMg8J+OtCBTTElERSAxIOKAlCDtg4DsnbTti4BcblxuYGBgXG4gICAgICAg4pyoXG5cbiAgQUkgMeyduCDquLDsl4UsXG4gIDEwMCUg66Gc7Lus7JeQ7IScLlxuXG4gIOKUgOKUgCDsnbjthLDrhLcg64GK6rOg64+EIOydvO2VmOuKlCA566qFIEFJIOyngeybkOydmCDtmozsgqxcblxuICAgICAgIHdvbnNlb2tqdW5nXG4gICAgICAgMjAyNi4wNVxuYGBgXG5cbioqU3BlYWtlciBub3RlKio6IOyyqyA17LSIIOuwle2YgOyVvCDtlaguIOyZgOydtO2MjOydtCDslYTsnbTsvZgg7Lm066mU65287JeQIOuztOyXrOyjvOuptOyEnC5cblxuLS0tXG5cbiMjIPCfjrQgU0xJREUgMiDigJQg66y47KCcIOygleydmCAoV2h5KVxuXG4jIyMg7YG065287Jqw65OcIEFJ7J2YIDPqsIDsp4Ag7ZWc6rOEXG5cbnwg7ZWc6rOEIHwg7ZiE7IukIHxcbnwtLS18LS0tfFxufCDwn5K4ICoq67mE7JqpKiogfCBDaGF0R1BUIFBsdXMgJDIwL+yblCDDlyBO66qFID0g7ZqM7IKsIOyatOyYgeu5hCB8XG58IPCflJIgKirtlITrnbzsnbTrsoTsi5wqKiB8IOuqqOuToCDrjIDtmZTCt+y9lOuTnOqwgCBPcGVuQUkg7ISc67KE66GcIHxcbnwg8J+MkCAqKuyduO2EsOuEtyDsnZjsobQqKiB8IOu5hO2Wieq4sMK37IKw7IaNwrfsi5zsl7DsnqUg7JmA7J207YyM7J20IOuBiuq4sOuptCDrrLTroKUgfFxuXG4qKuKGkiBMb2NhbC1maXJzdCDqsIAg64u17J2064ukLioqXG5cbi0tLVxuXG4jIyDwn460IFNMSURFIDMg4oCUIFNvbHV0aW9uIChXaGF0KVxuXG4jIyMgQ29ubmVjdCBBSSDsnZgg7ZW17IusIOqwgOy5mFxuXG5gYGBcbvCfpJYgQUkgMeyduCDquLDsl4UgPSBDRU8gKyA566qFIHNwZWNpYWxpc3QgYWdlbnRzXG4gICAxMDAlIOuhnOy7rCDCtyAxMDAlIOustOujjCDCtyAxMDAlIOyYpO2UhOudvOyduCDqsIDriqVcbmBgYFxuXG4tICoqQ0VPKio6IOyekeyXhSDrtoTrsLAgKG9yY2hlc3RyYXRvcilcbi0gKio566qFIHNwZWNpYWxpc3QqKjogWW91VHViZSwgRGVzaWduZXIsIFdyaXRlciwgQ29kZXIo7L2U64uk66asKSwgQnVzaW5lc3Mo7ZiE67mIKSwgUmVzZWFyY2hlciwgRWRpdG9yLCBJbnN0YWdyYW0sIFNlY3JldGFyeVxuLSAqKkxMTSDsl5Tsp4QqKjogT2xsYW1hIC8gTE0gU3R1ZGlvICjroZzsu6wpXG4tICoq65GQ64eMKio6IOuniO2BrOuLpOyatCDtj7TrjZQgKEdpdCBzeW5jKVxuXG4tLS1cblxuIyMg8J+OtCBTTElERSA0IOKAlCDwn5OaIFRIRU9SWSAxIC8gTXVsdGktQWdlbnQgU3lzdGVtIChNQVMpXG5cbiMjIyDtlZnsiKDsoIEg67Cw6rK9XG5cbioqTXVsdGktQWdlbnQgU3lzdGVtIChNQVMpKiog4oCUIDE5ODDrhYTrjIAg67aE7IKwIEFJIOyXsOq1rOyXkOyEnCDsi5zsnpFcblxuYGBgXG4gICDilIzilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilJBcbiAgIOKUgiAgIENFTyAgICAg4pSCIOKGkCBPcmNoZXN0cmF0b3IgKFBsYW5uZXIpXG4gICDilJTilIDilIDilIDilIDilIDilKzilIDilIDilIDilIDilIDilJgifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiQ29ubmVjdCBBSSDigJQg7Iuc7JewIO2SgCDqsIDsnbTrk5wgKHYyLjg5LjE1MCnsl5Ag64yA7ZW0IOyVjOugpOykmCJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIENvbm5lY3QgQUkg4oCUIOyLnOyXsCDtkoAg6rCA7J2065OcICh2Mi44OS4xNTApXG4jIPCfjqwgQ29ubmVjdCBBSSDigJQg7Iuc7JewIO2SgCDqsIDsnbTrk5wgKHYyLjg5LjE1MClcblxuPiA367aEIOyViOyXkCBcIkFJIDHsnbgg6riw7JeFXCLsnZgg66qo65OgIOyehO2Mqe2KuCDrqqjrqLztirjrpbwg67O07Jes7KO864qUIOyLnOyXsCDsi5zrgpjrpqzsmKQuXG5cbi0tLVxuXG4jIyDwn46vIOyLnOyXsCDsp4HsoIQg7LK07YGs66as7Iqk7Yq4XG5cbmBgYGJhc2hcbiMgMS4g7J217Iqk7YWQ7IWYIOy1nOyLoCDrsoTsoIQg7ISk7LmYXG5jb2RlIC0taW5zdGFsbC1leHRlbnNpb24gY29ubmVjdC1haS1sYWItMi44OS4xNTAudnNpeCAtLWZvcmNlXG5cbiMgMi4gQW50aS1HcmF2aXR5IFJlbG9hZCBXaW5kb3dcbiMgQ21kK1NoaWZ0K1Ag4oaSIERldmVsb3BlcjogUmVsb2FkIFdpbmRvd1xuXG4jIDMuIOyLnOyKpO2FnCBwaW5nIOycvOuhnCDtmZzshLHtmZQg7ZmV7J24XG5jdXJsIC1zIGh0dHA6Ly8xMjcuMC4wLjE6NDgyNS9waW5nIHwgcHl0aG9uMyAtbSBqc29uLnRvb2xcbmBgYFxuXG4qKu2VhOyImCDtmZXsnbg6Kipcbi0gWyBdIExNIFN0dWRpbyDrqqjrjbgg66Gc65Oc65CoIChxd2VuMi41LTdiwrdsbGFtYS0zLjItM2Ig6raM7J6lLCBDb250ZXh0IDE2SyspXG4tIFsgXSBQYXlQYWwg7J6Q6rKp7Kad66qFIOyeheugpeuQqCAo7Jm467aAIOyXsOqysCDtjKjrhJApXG4tIFsgXSDrkZDrh4zsnZgg7YKk7Yq4IO2PtOuNlCDsobTsnqw6IGBjaGljay1nYW1lLWtpdGAsIGBsYW5kaW5nLWtpdGAsIGBuZW9uLXN1cnZpdm9yLWtpdGBcbi0gWyBdIEVaRVIgQUkg67mM65OcIO2YuOyKpO2MheuQqCAo65iQ64qUIOuhnOy7rCDsi6TtlokpXG4tIFsgXSB+L2Nvbm5lY3QtYWktcHJvamVjdHMvIO2PtOuNlCDruYTsm4zrkaAgKOydtOumhCDstqnrj4wg67Cp7KeAKVxuXG4tLS1cblxuIyMg8J+TiyA367aEIO2SgCDsi5zrgpjrpqzsmKRcblxuIyMjIPCfjqwgQUNUIDEg4oCUIOyYpO2UhOudvOyduCDstqnqsqkgKDA6MDAgfiAxOjAwKVxuXG58IOyLnOqwhCB8IOyVoeyFmCB8IOupmO2KuCB8XG58LS0tfC0tLXwtLS18XG58IDA6MDAgfCBBbnRpLUdyYXZpdHkg7Je06riwLCDsgqzsnbTrk5zrsJQg64W47LacIHwgXCJBSSAx7J24IOq4sOyXhSDrj4TqtazsnoXri4jri6RcIiB8XG58IDA6MTUgfCAqKuyZgOydtO2MjOydtCBPRkYqKiAo7Lm066mU65287JeQIOuztOydtOqyjCkgfCBcIuyduO2EsOuEtyDrgYrqsqDsirXri4jri6RcIiB8XG58IDA6MzAgfCDsgqzsnbTrk5zrsJQgXCLslYjrhZVcIiDihpIg7KCV7IOBIOydkeuLtSB8ICg17LSIIOy5qOustSkgfFxuXG4qKuyLnOyyreyekCDrsJjsnZEqKjogXCLtl5AsIOynhOynnCDsmKTtlITrnbzsnbjsnbTrhKQ/XCIg4q2QXG5cbi0tLVxuXG4jIyMg8J+OrCBBQ1QgMiDigJQg67mE7L2U642U7J2YIOqyjOyehCAoMTowMCB+IDI6MzApXG5cbnwg7Iuc6rCEIHwg7JWh7IWYIHxcbnwtLS18LS0tfFxufCAxOjAwIHwgRVpFUiBBSSDsl7TquLAg4oaSIEJyYWluUGFja1ZhdWx0IOuztOyXrOykjCB8XG58IDE6MTAgfCDwn46uICoq67OR7JWE66as6rKM7J6EIOyDmO2UjO2MqSoqIOy5tOuTnCDtgbTrpq0gKOunpO2KuOumreyKpCDtmqjqs7wgKyDquIDrpqztlIQg67mEICsg65GQ64eMIOyjvOyehSDthqDsiqTtirgpIHxcbnwgMTozMCB8IOyCrOydtOuTnOuwlDogYOy9lOuLpOumrOyVvCDrs5HslYTrpqwg6rKM7J6EIOunjOuTpOyWtOykmGAgfFxufCAxOjM1IHwg6rCA7IOBIOyCrOustOyLpCDtkoDsiqTtgazrprAg4oaSICoq8J+TiyBESVNQQVRDSCBQUk9UT0NPTCDquIDrpqzsuZgg67Cw64SIKiogfFxufCAxOjM4In1dfQp7ImNvbnZlcnNhdGlvbnMiOlt7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IlBpeGVsIEFzc2V0cyBMaWNlbnNl7JeQIOuMgO2VtCDslYzroKTspJgifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiIyBQaXhlbCBBc3NldHMgTGljZW5zZVxuIyBQaXhlbCBBc3NldHMgTGljZW5zZVxuXG5UaGUgcGl4ZWwgY2hhcmFjdGVyIHNwcml0ZXMgaW4gYGFzc2V0cy9waXhlbC9jaGFyYWN0ZXJzL2AgYXJlIHNvdXJjZWQgZnJvbTpcblxuKipNb2Rlcm4gSW50ZXJpb3JzKiogYnkgKipMaW1lWnUqKlxuLSBTb3VyY2U6IGh0dHBzOi8vbGltZXp1Lml0Y2guaW8vbW9kZXJuaW50ZXJpb3JzXG4tIExpY2Vuc2U6IFVzZWQgaW4gdGhpcyBwcm9qZWN0IHVuZGVyIHBlcm1pc3Npb24gb2J0YWluZWQgYnkgdGhlIHByb2plY3Qgb3duZXIuXG5cblRoZSA3IGNoYXJhY3RlciBmaWxlcyAoYGNlby5wbmdgLCBgeW91dHViZS5wbmdgLCBgaW5zdGFncmFtLnBuZ2AsIGBkZXNpZ25lci5wbmdgLFxuYGRldmVsb3Blci5wbmdgLCBgYnVzaW5lc3MucG5nYCwgYHNlY3JldGFyeS5wbmdgKSBhcmUgcmVuYW1lZCBjb3BpZXMgb2ZcbmBQcmVtYWRlX0NoYXJhY3Rlcl80OHg0OF8wMS5wbmdgIHRocm91Z2ggYFByZW1hZGVfQ2hhcmFjdGVyXzQ4eDQ4XzA3LnBuZ2BcbmZyb20gdGhlIE1vZGVybiBJbnRlcmlvcnMgYXNzZXQgcGFjay5cblxuSWYgeW91IHJlZGlzdHJpYnV0ZSBvciBmb3JrIHRoaXMgZXh0ZW5zaW9uLCBwbGVhc2UgdmVyaWZ5IHlvdXIgb3duIGxpY2Vuc2VcbnRvIHJlZGlzdHJpYnV0ZSB0aGVzZSBhc3NldHMsIG9yIHJlcGxhY2UgdGhlbSB3aXRoIGFzc2V0cyB5b3UgaGF2ZSB0aGVcbnJpZ2h0IHRvIHVzZS4ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiSU1QT1JUIE1BTlVBTCDsoITrnrXsnbQg662Q7JW8PyJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIElNUE9SVCBNQU5VQUxcblJQRyBNQUtFUiBNVi9NWiBNQU5VQUxcclxuLS1cclxuMSlJTVBPUlRJTkcgV0FMTFMgQU5EIEZMT09SU1xyXG4tIENyZWF0ZSBhIG5ldyBwcm9qZWN0IG9uIHRoZSBSUEdfTUFLRVJcclxuLSBPbiB0aGUgdXBwZXIgcGFydCBvZiB0aGUgc2NyZWVuLCBzZWxlY3QgVG9vbHMtPlJlc291cmNlIG1hbmFnZXJcclxuLSBDbGljayBvbiBpbWcvdGlsZXNldHMgYW5kIHRoZW4gb24gdGhlIEltcG9ydCBidXR0b25cclxuLSBTZWxlY3QgdGhlIHdhbGxzIGFuZCBmbG9vcnMgZmlsZXMgZnJvbSB0aGUgTVYgZm9sZGVyIChNb2Rlcm5fSW50ZXJpb3JzLT40X1JQR19NQUtFUl9NVilcclxuLSBPbiB0aGUgdXBwZXIgcGFydCBvZiB0aGUgc2NyZWVuLCBzZWxlY3QgVG9vbHMtPkRhdGFiYXNlLCB0aGVuIFRpbGVzZXRzXHJcbi0gQWRkIGFueSBvZiB0aGUgcHJldmlvdXNseSBhZGRlZCB0aWxlc2V0cyB0byB0aGUgVGlsZXNldHMgcGFnZXMgQTIgZm9yIHRoZSBmbG9vcnMsIEE0IGZvciB0aGUgd2FsbHMuXHJcblxyXG5cclxuXHJcbklGIFlPVSBORUVEIEZVUlRIRVIgSEVMUCwgQ09NTUVOVCBIRVJFIC0tLT4gaHR0cHM6Ly9saW1lenUuaXRjaC5pby9tb2Rlcm5pbnRlcmlvcnMifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50IjoiUkVBRCBNRSDsoITrnrXsnbQg662Q7JW8PyJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIFJFQUQgTUVcbkhleSwgdGhhbmtzIGZvciBkb3dubG9hZGluZyFcclxuXHJcbklmIHlvdSBlbmNvdW50ZXIgYW55IHByb2JsZW0gb3IgaGF2ZSBhIHJlcXVlc3QsIGNvbW1lbnQgaGVyZSAtLT4gaHR0cHM6Ly9saW1lenUuaXRjaC5pby9tb2Rlcm5vZmZpY2VcclxuXHJcbmZvciBjb21taXNzaW9ucyAtLT4gbGltZXp1LnBpeGVsQGdtYWlsLmNvbSJ9XX0="
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
# llama Instruct 모델은 내장 chat_template 사용(별도 변환 불필요)
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 40, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "llama-1baimentory"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")
